In [ ]:
!pip install qiskit
!pip install qiskit-aer
!pip install qiskit-ibm-runtime
!pip install cirq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.8/212.8 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 11.4 MB/s eta 0:00:00


In [ ]:
import numpy as np
from numpy import pi
import random
from functools import partial
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import eigh, sqrtm
from scipy.special import erf
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit import Parameter
from qiskit.circuit.classical import expr
from qiskit.quantum_info import SparsePauliOp, Statevector, Operator, random_statevector, process_fidelity
from qiskit.circuit.library import QAOAAnsatz, hamiltonian_variational_ansatz, XXPlusYYGate, CPhaseGate, UnitaryGate, U3Gate, CXGate,PauliEvolutionGate
from qiskit.synthesis import TwoQubitBasisDecomposer
from qiskit.synthesis.evolution import SuzukiTrotter
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler, EstimatorV2 as Estimator,QiskitRuntimeService
from qiskit_ibm_runtime.options import EnvironmentOptions, EstimatorOptions,SamplerOptions

import cirq

/usr/local/lib/python3.12/dist-packages/samplomatic/__init__.py:20: UserWarning: 
You have imported samplomatic==0.18.0 which is in 
beta development. Please expect breaking changes between 
minor versions and pin your dependencies accordingly.
  _warn_once_per_version(


## Measurement


In [ ]:
def measure_ZZ(qc, q0, q1, cbit):
    qc.cx(q0, q1)
    qc.measure(q1, cbit)
    qc.cx(q0, q1)

In [ ]:
def measure_XI(qc, q0, q1, cbit):
    qc.h(q0)
    qc.measure(q0, cbit)
    qc.h(q0)

**Important: fix convention $ Y = S X S^\dagger$. Later formulas must use the same convention to ensure a correct pauli tracking**

In [ ]:
def measure_YI(qc, q0, q1, cbit):
    qc.sdg(q0)
    measure_XI(qc, q0, q1, cbit)
    qc.s(q0)

In [ ]:
def measure_ZY(qc, q0, q1, cbit):

    # with the same convention, Y = S X S^d = S H Z H S^d
    qc.sdg(q1)
    qc.h(q1)
    measure_ZZ(qc, q0, q1, cbit)
    qc.h(q1)
    qc.s(q1)

In [ ]:
def measure_ZX(qc, q0, q1, cbit):

    qc.h(q1)
    measure_ZZ(qc, q0, q1, cbit)
    qc.h(q1)

:## Single qubit clifford group

In [ ]:
def add_H(qc, d, a, cbit):
    c = cbit

    measure_XI(qc, a, d, c[0])
    measure_ZY(qc, a, d, c[1])
    measure_YI(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])

    parity = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[2])
    with qc.if_test(expr.logic_not(parity)):
        qc.y(d)

    qc.x(d)

    qc.reset(a) # easy to check with gate-based circuit

    return qc

In [ ]:
def add_S(qc, d, a, cbit):
    c = cbit

    # (ancilla, data) = (q+1, q)

    measure_XI(qc, a, d, c[0])
    measure_ZZ(qc, a, d, c[1])
    measure_YI(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])

    # measurement bit c_i encodes s_i = (-1)^{c_i}
    # s_i s_j = (-1)^{c[i] + c[j]}
    # product becomes XOR

    # Z^{(1 + s0 s1 s2)/2}
    # exponent = 1 when s0 s1 s2 = +1
    # s0 s1 s2 = (-1)^{c0 + c1 + c2}
    # +1 when (c0 + c1 + c2) mod 2 = 0 (even parity)

    parity = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[2])
    with qc.if_test(expr.logic_not(parity)):
        qc.z(d)

    qc.reset(a) # easy to check with gate-based circuit

    return qc

In [ ]:
def add_SH(qc, d, a, cbit):
    c = cbit

    measure_XI(qc, a, d, c[0])  # s0
    measure_ZZ(qc, a, d, c[1])  # s1
    measure_ZY(qc, a, d, c[2])  # s2
    measure_YI(qc, a, d, c[3])  # s3
    measure_XI(qc, a, d, c[4])  # s4

    # X^{(1 + s0 s2 s3)/2} even parity
    parity_023 = expr.bit_xor(expr.bit_xor(c[0], c[2]), c[3])
    with qc.if_test(parity_023):
        qc.y(d)

    # Z^{(1 - s1 s2)/2} even parity
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(parity_12):
        qc.z(d)

    qc.reset(a)

    return qc

In [ ]:
def add_HSH(qc, d, a, cbit):
    c = cbit

    measure_XI(qc, a, d, c[0])
    measure_ZZ(qc, a, d, c[1])
    measure_ZY(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])

    # Y^{(1 - s0 s3)/2} odd parity
    parity_03 = expr.bit_xor(c[0], c[3])
    with qc.if_test(parity_03):
        qc.y(d)

    # X^{(1 + s1 s2)/2} even parity
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(d)

    qc.reset(a)

    return qc

In [ ]:
def add_HS(qc, d, a, cbit):
    c = cbit

    measure_XI(qc, a, d, c[0])  # s0
    measure_ZY(qc, a, d, c[1])  # s1
    measure_ZZ(qc, a, d, c[2])  # s2
    measure_YI(qc, a, d, c[3])  # s3
    measure_XI(qc, a, d, c[4])  # s4

    # X^{(1 + s1 s2)/2} even parity
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(d)

    # Z^{(1 + s0 s1 s3)/2} even parity
    parity_013 = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[3])
    with qc.if_test(expr.logic_not(parity_013)):
        qc.z(d)

    qc.reset(a)

    return qc

In [ ]:
def sqrt_W():
    X = np.array([
        [0, 1],
        [1, 0]
    ], dtype=complex)

    Y = np.array([
        [0, -1j],
        [1j, 0]
    ], dtype=complex)

    W = (X + Y) / np.sqrt(2)

    return sqrtm(W)

## Single-qubit gate in supremacy circuit

In [ ]:
def add_sqrtX(qc, d, a, cbit):
    qc = add_HSH(qc, d, a, cbit)
    return qc

In [ ]:
def add_sqrtY(qc, d, a, cbit):
   qc = add_S(qc, d, a, cbit)
   qc = add_HS(qc, d, a, cbit)
   return qc

In [ ]:
def add_sqrtW(qc, d, a, cbit):
    qc.tdg(d)
    qc = add_sqrtX(qc, d, a, cbit)
    qc.t(d)
    return qc

## Two-qubit gate in supremacy circuit

In [ ]:
# fSim matrix

def fSim(theta, phi):
    return np.array([
        [1, 0, 0, 0],
        [0, np.cos(theta), -1j*np.sin(theta), 0],
        [0, -1j*np.sin(theta), np.cos(theta), 0],
        [0, 0, 0, np.exp(-1j*phi)]
    ], dtype=complex)

In [ ]:
def add_CNOT(qc, c, a, t, cbit):
    # needs 8 cbits. all_ reuses cbit[0-4]. measure_ uses cbit[5-7]
    # Qubit A is initialized in an eigenstate of Z

    # prepare a in z-basis
    qc.reset(a)

    measure_ZX(qc, c, a, cbit[5])
    measure_ZX(qc, a, t, cbit[6])

    # this is single X-measurement on qubit a. h is on level of simulation, not circuit element
    qc.h(a)
    qc.measure(a, cbit[7])
    qc.h(a)

    # reset a to 0 for an easier comparision
    qc.reset(a)

    # we use cbit directly because it stores as 0(even) and 1(odd), as needed in paper
    P1 = cbit[5]
    P2 = cbit[6]
    M  = cbit[7]

    # X_t^{(P1 ⊕ M)}
    xt = expr.bit_xor(P1, M)
    with qc.if_test(xt):
        qc.x(t)

    #Z_c^{P2}
    with qc.if_test((P2, 1)):
        qc.z(c)

    return qc

In [ ]:
def apply_rz_or_s(qc, qubit, angle):

    # identify z-rotation as clifford gate

    # normalize to [-π, π]
    theta = (angle + np.pi) % (2*np.pi) - np.pi

    # check multiple of π/2
    k = round(theta / (np.pi/2))

    if abs(theta - k*(np.pi/2)) < 1e-3:
        k_mod = k % 4
        for _ in range(k_mod):
            qc.s(qubit)
    else:
        qc.rz(theta, qubit)

    return qc

In [ ]:
def get_zsx_blocks_and_cx(theta, phi):

    # decompose fSim into 3 CNOTs and Euler rotations in ZSX basis

    qc = QuantumCircuit(2)
    qc.append(UnitaryGate(fSim(theta, phi)), [0, 1])

    U_target = Operator(qc).data

    decomposer = TwoQubitBasisDecomposer(
        CXGate(),
        euler_basis="ZSX",
    )

    synth = decomposer(U_target)

    synth = transpile(
        synth,
        basis_gates=["rz", "sx", "x", "cx"],
        optimization_level=3,
    )

    blocks = []
    cx_dirs = []
    current = {0: [], 1: []}

    for instr in synth.data:
        inst = instr.operation
        qargs = instr.qubits
        name = inst.name

        if name == "cx":
            blocks.append(current)
            current = {0: [], 1: []}

            ctrl = synth.find_bit(qargs[0]).index
            targ = synth.find_bit(qargs[1]).index
            cx_dirs.append((ctrl, targ))

            continue

        q = synth.find_bit(qargs[0]).index

        if name == "rz":
            current[q].append(("rz", float(inst.params[0])))
        elif name == "sx":
            current[q].append(("sx", None))
        elif name == "x":
            current[q].append(("x", None))

    blocks.append(current)

    return blocks, cx_dirs

In [ ]:
def add_TwoQubitGate(qc, c, a, t, theta, phi, Z1, Z2, Z3, Z4, cbit):

    # build two qubit gate for supremacy circuit

    qc.rz(Z1, c)
    qc.rz(Z2, t)

    blocks, cx_dirs = get_zsx_blocks_and_cx(theta, phi)

    for layer in range(4):

        block = blocks[layer]

        # ---- 1Q gates ----
        for qubit in [0, 1]:

            target = c if qubit == 0 else t

            for name, angle in block[qubit]:

                if name == "rz":
                    qc = apply_rz_or_s(qc, target, angle)

                elif name == "sx":
                    qc = add_sqrtX(qc, target, a, cbit)

                elif name == "x":
                    qc = add_sqrtX(qc, target, a, cbit)
                    qc = add_sqrtX(qc, target, a, cbit)

        # ---- CX with correct direction ----
        if layer < 3:
            ctrl, targ = cx_dirs[layer]

            if ctrl == 0:
                qc = add_CNOT(qc, c, a, t, cbit)
            else:
                qc = add_CNOT(qc, t, a, c, cbit)

    qc.rz(Z3, c)
    qc.rz(Z4, t)

    return qc

# 12-Qubit Supremacy Circuit

We use only **1 ancilla qubit**.  
This **12+1** construction is equivalent to the original **12+12 tetron layout** due to:

- ancilla qubits form a product state instantanouesly in **noiseless** condition (*correct physics*),
- and successful MVP circuit test *(correct implementation)*.

The initial state is the classical bitstring: $|00\cdots0\rangle$

The circuit data is stored in: **circuit_n12_m14_s0_e0_pEFGH_snapped.py**

The wave function is stored in: **amplitudes_n12_m14_s0_e0_pEFGH.txt**


## Important note:

In Google's simulation, the qubits order is

```python
QUBIT_ORDER = [
    cirq.GridQubit(3, 3),
    cirq.GridQubit(3, 4),
    cirq.GridQubit(3, 5),
    cirq.GridQubit(3, 6),
    cirq.GridQubit(4, 3),
    cirq.GridQubit(4, 4),
    cirq.GridQubit(4, 5),
    cirq.GridQubit(4, 6),
    cirq.GridQubit(5, 3),
    cirq.GridQubit(5, 4),
    cirq.GridQubit(5, 5),
    cirq.GridQubit(5, 6),
]
```
, which implies

$$
(3,3)=0,\quad (3,4)=1,\quad \dots,\quad (5,6)=11.
$$

This indexing convention **does not match the experimental layout**. For example, in the experimental device labeling, qubit $(3,3)$ is labeled as qubit $11$.

Also, **Cirq uses the reverse of Qiskit's qubit ordering.**

In [ ]:
# 12+1 supremacy MBQC

def Supremacy_MBQC_trick(document):

    # =========================================================
    # Execute the file so we obtain:
    #   QUBIT_ORDER
    #   CIRCUIT
    # =========================================================

    namespace = {}

    with open(document, "r") as f:
        code = f.read()

    exec(code, namespace)

    QUBIT_ORDER = namespace["QUBIT_ORDER"]
    CIRCUIT = namespace["CIRCUIT"]

    # Map GridQubit -> integer index
    qubit_map = {q: i for i, q in enumerate(QUBIT_ORDER)}
    N = len(QUBIT_ORDER)

    # one ancilla qubit for noiseless simulation
    anc = N
    qc = QuantumCircuit(N + 1)

    # classical register
    if not qc.cregs:
        qc.add_register(ClassicalRegister(8))

    cbit = qc.cregs[0]

    # =========================================================
    # Walk through every Cirq operation
    # =========================================================

    for moment in CIRCUIT:

        for op in moment.operations:

            gate = op.gate
            qubits = op.qubits
            # qubit indices
            q_idx = [qubit_map[q] for q in qubits]

            # sqrt(X)
            if isinstance(gate, cirq.XPowGate):

                if np.isclose(gate.exponent, 0.5):
                    qc = add_sqrtX(qc, q_idx[0], anc, cbit)

            # sqrt(Y)
            elif isinstance(gate, cirq.YPowGate):
                if np.isclose(gate.exponent, 0.5):
                    qc = add_sqrtY(qc, q_idx[0], anc, cbit)


            # sqrt(W)
            elif isinstance(gate, cirq.PhasedXPowGate):

                if (np.isclose(gate.phase_exponent, 0.25) and np.isclose(gate.exponent, 0.5)):

                    qc = add_sqrtW(qc, q_idx[0], anc, cbit)

            # Rz
            elif isinstance(gate, cirq.ZPowGate):
                radians = gate.exponent * np.pi
                qc.rz(radians, q_idx[0])

            # FSim
            elif isinstance(gate, cirq.FSimGate):
                theta = gate.theta
                phi = gate.phi
                qc = add_TwoQubitGate(qc, q_idx[0], anc, q_idx[1],theta, phi, 0, 0, 0, 0, cbit) # this function has built-in Rz. Becuase I already use Rz, just set them to 0

        qc.barrier()

    return qc

In [ ]:
# read Google's wave function

sv_file = np.zeros(2**12, dtype=complex)

with open("/content/drive/My Drive/microsoft/amplitudes_n12_m14_s0_e0_pEFGH.txt") as f:
    for line in f:
        bits, re, im = line.split()

        amp = float(re) + 1j * float(im)

        # reverse because external convention != Qiskit convention
        idx = int(bits[::-1], 2)

        sv_file[idx] = amp

In [ ]:
document = "/content/drive/My Drive/microsoft/circuit_n12_m14_s0_e0_pEFGH.py"

qc_mbqc = Supremacy_MBQC_trick(document)
qc_mbqc.save_statevector(conditional=True)

sim = AerSimulator(method="statevector")

result = sim.run(qc_mbqc, shots=1).result()
data = result.data(0)["statevector"]
sv_mbqc = list(data.values())[0]

# the ancilla qubit is in |0>. Just pick half of wavefunction to trace out

vec0 = sv_mbqc.data[:2**12]
vec1 = sv_mbqc.data[2**12:]

if np.linalg.norm(vec0) >= np.linalg.norm(vec1):
    sv_mbqc = vec0 / np.linalg.norm(vec0)
else:
    sv_mbqc = vec1 / np.linalg.norm(vec1)


overlap = abs(np.vdot(sv_mbqc.data, sv_file))
print(f"overlap = {abs(np.vdot(sv_mbqc.data, sv_file))}")

tol = 1e-6
if 1-overlap <= tol:
    print("✅ success")
else:
    print("❌ fail")

overlap = 0.9999996754047306
✅ success


In [ ]:
qc_mbqc.draw(fold=-1)

┌─────┐        ┌───┐┌─┐┌───┐┌─────┐┌───┐┌───┐┌─┐┌───┐┌───┐┌───┐     ┌──────────────────── ┌───┐ ───────┐ ┌─────────────────────── ┌───┐ ───────┐ ┌───┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  ░                  ░                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

### Gate based circuit as a reference

In [ ]:
def Supremacy_gate(document):

    # =========================================================
    # Execute the file so we obtain:
    #   QUBIT_ORDER
    #   CIRCUIT
    # =========================================================

    namespace = {}

    with open(document, "r") as f:
        code = f.read()

    exec(code, namespace)

    QUBIT_ORDER = namespace["QUBIT_ORDER"]
    CIRCUIT = namespace["CIRCUIT"]

    # Map GridQubit -> integer index

    qubit_map = {q: i for i, q in enumerate(QUBIT_ORDER)}
    N = len(QUBIT_ORDER)
    qc = QuantumCircuit(N)

    # =========================================================
    # Walk through every Cirq operation
    # =========================================================

    for moment in CIRCUIT:

        for op in moment.operations:

            gate = op.gate
            qubits = op.qubits

            # qubit indices
            q_idx = [qubit_map[q] for q in qubits]

            # sqrt(X)
            if isinstance(gate, cirq.XPowGate):
                if np.isclose(gate.exponent, 0.5):
                    qc.sx(q_idx[0])

            # sqrt(Y)
            elif isinstance(gate, cirq.YPowGate):
                if np.isclose(gate.exponent, 0.5):
                    qc.ry(np.pi / 2, q_idx[0])

            # sqrt(W)
            elif isinstance(gate, cirq.PhasedXPowGate):
                if (np.isclose(gate.phase_exponent, 0.25) and np.isclose(gate.exponent, 0.5)):
                    qc.append(UnitaryGate(sqrt_W()), [q_idx[0]])

            # Rz
            elif isinstance(gate, cirq.ZPowGate):
                radians = gate.exponent * np.pi
                qc.rz(radians, q_idx[0])

            # FSim
            elif isinstance(gate, cirq.FSimGate):
                theta = gate.theta
                phi = gate.phi
                qc.append(UnitaryGate(fSim(theta, phi)),[q_idx[0], q_idx[1]])

        qc.barrier()

    return qc

In [ ]:
sv_gate = Statevector.from_instruction(qc_gate)
abs(np.vdot(sv_gate, sv_file))

np.float64(0.9999996754697728)

In [ ]:
document = "/content/drive/My Drive/microsoft/circuit_n12_m14_s0_e0_pEFGH.py"

qc_gate = Supremacy_gate(document)
qc_gate.draw(fold=-1)

┌─────────┐ ░                  ░              ░                 ░ ┌─────────┐ ░  ┌────────────┐ ░ ┌──────────┐ ░ ┌─────────────┐ ░    ┌────┐   ░                 ░                                                  ░                 ░ ┌─────────┐ ░  ┌───────────┐  ░ ┌──────────┐                                     ░ ┌─────────────┐ ░    ┌────┐   ░                 ░              ░                 ░ ┌─────────┐ ░  ┌────────────┐ ░ ┌──────────┐ ░ ┌─────────────┐ ░    ┌────┐   ░                 ░                                                  ░                 ░ ┌─────────┐ ░  ┌────────────┐ ░ ┌──────────┐                                     ░ ┌─────────────┐ ░    ┌────┐   ░                 ░              ░                 ░ ┌─────────┐ ░  ┌────────────┐ ░ ┌──────────┐ ░ ┌─────────────┐ ░ ┌─────────┐ ░                 ░                                                  ░                 ░ ┌─────────┐ ░  ┌───────────┐  ░ ┌──────────┐                                     ░ ┌─────────────┐ ░    ┌────┐   ░                 ░              ░                 ░ ┌─────────┐ ░  ┌────────────┐ ░ ┌──────────┐ ░ ┌─────────────┐ ░ ┌─────────┐ ░ 
 q_0: ┤ Unitary ├─░──────────────────░──────────────░─────────────────░─┤ Ry(π/2) ├─░──┤ Rz(7.9588) ├─░─┤0         ├─░─┤ Rz(-7.3704) ├─░────┤ √X ├───░─────────────────░──────────────────────────────────────────────────░─────────────────░─┤ Unitary ├─░──┤ Rz(39.91) ├──░─┤0         ├─────────────────────────────────────░─┤ Rz(-39.198) ├─░────┤ √X ├───░─────────────────░──────────────░─────────────────░─┤ Unitary ├─░──┤ Rz(23.767) ├─░─┤0         ├─░─┤ Rz(-23.179) ├─░────┤ √X ├───░─────────────────░──────────────────────────────────────────────────░─────────────────░─┤ Unitary ├─░──┤ Rz(81.756) ├─░─┤0         ├─────────────────────────────────────░─┤ Rz(-81.044) ├─░────┤ √X ├───░─────────────────░──────────────░─────────────────░─┤ Ry(π/2) ├─░──┤ Rz(39.576) ├─░─┤0         ├─░─┤ Rz(-38.987) ├─░─┤ Unitary ├─░─────────────────░──────────────────────────────────────────────────░─────────────────░─┤ Ry(π/2) ├─░──┤ Rz(123.6) ├──░─┤0         ├─────────────────────────────────────░─┤ Rz(-122.89) ├─░────┤ √X ├───░─────────────────░──────────────░─────────────────░─┤ Ry(π/2) ├─░──┤ Rz(55.384) ├─░─┤0         ├─░─┤ Rz(-54.796) ├─░─┤ Unitary ├─░─
      ├─────────┤ ░  ┌────────────┐  ░ ┌──────────┐ ░ ┌─────────────┐ ░ └──┬────┬─┘ ░ ┌┴────────────┤ ░ │  Unitary │ ░ └┬────────────┤ ░ ┌──┴────┴─┐ ░                 ░                                                  ░                 ░ ├─────────┤ ░  ├───────────┴┐ ░ │          │┌──────────┐                         ░ ├─────────────┤ ░    ├────┤   ░  ┌────────────┐ ░ ┌──────────┐ ░ ┌─────────────┐ ░ ├─────────┤ ░ ┌┴────────────┤ ░ │  Unitary │ ░ └┬────────────┤ ░    ├────┤   ░                 ░                                                  ░                 ░ ├─────────┤ ░  ├────────────┤ ░ │          │┌──────────┐                         ░ ├─────────────┤ ░    ├────┤   ░  ┌────────────┐ ░ ┌──────────┐ ░ ┌─────────────┐ ░ ├─────────┤ ░ ┌┴────────────┤ ░ │  Unitary │ ░ └┬────────────┤ ░ └──┬────┬─┘ ░                 ░                                                  ░                 ░ ├─────────┤ ░  ├───────────┴┐ ░ │          │┌──────────┐                         ░ └┬────────────┤ ░    ├────┤   ░   ┌──────────┐  ░ ┌──────────┐ ░ ┌─────────────┐ ░ ├─────────┤ ░  ├───────────┬┘ ░ │  Unitary │ ░ └┬───────────┬┘ ░ └──┬────┬─┘ ░ 
 q_1: ┤ Unitary ├─░──┤ Rz(0.8694) ├──░─┤0         ├─░─┤ Rz(-2.1118) ├─░────┤ √X ├───░─┤ Rz(-7.7748) ├─░─┤1         ├─░──┤ Rz(7.5543) ├─░─┤ Ry(π/2) ├─░─────────────────░──────────────────────────────────────────────────░─────────────────░─┤ Unitary ├─░──┤ Rz(38.278) ├─░─┤          ├┤0         ├─────────────────────────░─┤ Rz(-35.534) ├─░────┤ √X ├───░──┤ Rz(16.213) ├─░─┤0         ├─░─┤ Rz(-17.455) ├─░─┤ Unitary ├─░─┤ Rz(-23.583) ├─░─┤1         ├─░──┤ Rz(23.363) ├─░────┤ √X ├───░─────────────────░──────────────────────────────────────────────────░───────────